# 🎣 Phishing Detection & Email Threat Analysis

**Dataset:** 326 labeled real-world email samples  
**Goal:** Classify emails as Safe / Phishing, identify indicators, and build security awareness

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import re
from collections import Counter

# Load dataset
df = pd.read_csv('../dataset/curated_set.csv')

print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

## 2. Dataset Overview

In [ ]:
# Class distribution
counts = df['is_phishing'].value_counts()
print('Email Classification:')
print(f"  Legitimate (Safe): {counts[0]} emails ({counts[0]/len(df)*100:.1f}%)")
print(f"  Phishing:          {counts[1]} emails ({counts[1]/len(df)*100:.1f}%)")
print(f"  Total:             {len(df)} emails")
print()
print('Data Sources:')
print(df['source'].value_counts())

## 3. Phishing Indicator Detection

In [ ]:
def detect_indicators(text):
    """Detect common phishing indicators in email text."""
    text_lower = text.lower()
    indicators = {}

    # Urgency language
    urgency_words = ['immediately', 'urgent', 'suspended', 'act now', 'verify now',
                     'click here', 'limited time', 'warning', 'attention!!!', 'failure to']
    indicators['urgency_language'] = any(w in text_lower for w in urgency_words)

    # Suspicious URLs
    url_pattern = r'https?://[^\s]+'
    urls = re.findall(url_pattern, text)
    suspicious_domains = ['webs.com', 'sendspace.com', 'login./','login/login']
    indicators['suspicious_url'] = any(d in u for u in urls for d in suspicious_domains)
    indicators['has_url'] = len(urls) > 0

    # Credential request
    cred_words = ['verify', 'password', 'login', 'log on', 'confirm your', 'update your',
                  'sign in', 'recover', 'activate']
    indicators['credential_request'] = any(w in text_lower for w in cred_words)

    # Generic greeting (using email address as name)
    indicators['generic_greeting'] = bool(re.search(r'dear\s+[\w.]+@[\w.]+', text_lower))

    # Filter evasion (deliberate spacing/dots in words)
    indicators['filter_evasion'] = bool(re.search(r'\b\w+\s\.\s*\w+\b|\w\.\w\.\w', text))

    # COVID/crisis lure
    crisis_words = ['covid', 'pandemic', 'coronavirus', 'financial aid', 'unemployment benefit',
                    'financial hardship']
    indicators['crisis_lure'] = any(w in text_lower for w in crisis_words)

    # Impersonation signals
    impersonation = ['hr department', 'help desk', 'it support', 'admin', 'apple support',
                     'its service', 'mail delivery']
    indicators['impersonation'] = any(w in text_lower for w in impersonation)

    # Score: sum of True indicators
    indicators['risk_score'] = sum(1 for v in indicators.values() if v is True)

    return indicators

# Apply to all emails
indicator_df = df['text'].apply(detect_indicators).apply(pd.Series)
full_df = pd.concat([df, indicator_df], axis=1)

print('Indicator columns added:')
print(indicator_df.columns.tolist())

## 4. Indicator Frequency Analysis

In [ ]:
indicator_cols = ['urgency_language', 'suspicious_url', 'has_url', 'credential_request',
                  'generic_greeting', 'filter_evasion', 'crisis_lure', 'impersonation']

phishing = full_df[full_df['is_phishing'] == 1]
safe     = full_df[full_df['is_phishing'] == 0]

print('Indicator Prevalence (% of emails where indicator is present)\n')
print(f"{'Indicator':<25} {'Phishing':>12} {'Safe':>12}")
print('-' * 52)
for col in indicator_cols:
    p_pct = phishing[col].mean() * 100
    s_pct = safe[col].mean() * 100
    print(f"{col:<25} {p_pct:>11.1f}% {s_pct:>11.1f}%")

## 5. Email Classification Function

In [ ]:
def classify_email(text):
    """Classify an email as Safe, Suspicious, or Phishing with explanation."""
    indicators = detect_indicators(text)
    score = indicators['risk_score']
    found = [k for k, v in indicators.items() if v is True and k != 'risk_score']

    if score == 0:
        classification = '✅ SAFE'
    elif score <= 2:
        classification = '⚠️  SUSPICIOUS'
    else:
        classification = '🚨 PHISHING'

    print(f'Classification: {classification}')
    print(f'Risk Score:     {score}/7')
    print(f'Indicators:     {", ".join(found) if found else "None"}')
    print(f'\nEmail Preview:\n{text[:250]}...' if len(text) > 250 else f'\nEmail:\n{text}')
    return classification

# Test with examples from the dataset
print('=== EXAMPLE 1 ===')
classify_email(df[df['is_phishing']==1]['text'].iloc[5])
print()
print('=== EXAMPLE 2 (SAFE) ===')
classify_email(df[df['is_phishing']==0]['text'].iloc[2])

## 6. Top Phishing Techniques Summary

In [ ]:
techniques = {
    'Credential Harvesting': 'Fake login pages designed to steal usernames and passwords',
    'Urgency / Social Engineering': 'Artificial time pressure to bypass rational thinking',
    'Brand Impersonation': 'Mimicking Apple, HR, universities to appear legitimate',
    'Filter Evasion': 'Deliberate misspellings ("me ss ages") to bypass spam filters',
    'COVID-19 / Crisis Lures': 'Exploiting pandemic anxiety with fake financial aid or job offers',
}

print('TOP PHISHING TECHNIQUES IDENTIFIED\n')
print(f"{'Technique':<30} {'Description'}")
print('=' * 80)
for i, (tech, desc) in enumerate(techniques.items(), 1):
    print(f"{i}. {tech:<28} {desc}")

## 7. Prevention Checklist

In [ ]:
prevention_tips = [
    ('CRITICAL', 'Never click links in unexpected emails — type URLs manually'),
    ('CRITICAL', 'Check the full sender email address, not just the display name'),
    ('CRITICAL', 'Never share credentials via email — no legit org will ask'),
    ('CRITICAL', 'Enable Multi-Factor Authentication (MFA) on all accounts'),
    ('HIGH',     'Hover over links to preview the destination URL before clicking'),
    ('HIGH',     'Report suspicious emails to your security team immediately'),
    ('HIGH',     'If an email creates panic — STOP. Verify through another channel'),
    ('MEDIUM',   'Keep browser and email client software up to date'),
    ('MEDIUM',   'Attend regular security awareness training'),
]

print('PHISHING PREVENTION CHECKLIST\n')
for priority, tip in prevention_tips:
    icon = '🔴' if priority == 'CRITICAL' else '🟡' if priority == 'HIGH' else '🟢'
    print(f'{icon} [{priority:<8}] {tip}')

---
## 📄 Full Report

See `../report/Phishing_Detection_Awareness_Report.docx` for the complete professional report including:
- Annotated phishing email examples
- Risk explanation tables
- Prevention guidelines
- STOP-THINK-VERIFY framework
